# 005 · Slicing

> **Syntax › Collections** · Python 3.14

Slicing extracts a sub-sequence with `seq[start:stop:step]`. It works on every built-in sequence — lists, strings, tuples, `range`, `bytes` — and is one of the most-used pieces of Python syntax. C# 8 ranges (`a[1..^1]`) cover part of the same ground.

**In this notebook you will learn:**

- the `start:stop:step` syntax and its defaults
- negative indexes and negative steps (including `[::-1]` to reverse)
- how slicing compares to C# ranges and `Span<T>` — and when a copy is made
- slice assignment and deletion on lists
- `slice()` objects, and why out-of-range slices don't raise

## C# mental model

| C# | Python |
|---|---|
| `a[1..4]` | `a[1:4]` |
| `a[..3]` | `a[:3]` |
| `a[2..]` | `a[2:]` |
| `a[^2..]` | `a[-2:]` |
| `a[1..^1]` | `a[1:-1]` |
| `a[..]` | `a[:]` |
| *(no step)* | `a[::2]`, `a[::-1]` |
| `new Range(1, 4)` | `slice(1, 4)` |

Both languages use a **half-open** interval: `start` is included, `stop` is excluded. The main differences: Python has a **step**, Python **never throws** on out-of-range slice bounds, and Python has no built-in zero-copy `Span<T>` for lists.

## Basic slicing: `start:stop`

```csharp
// C#
int[] a = [10, 20, 30, 40, 50, 60];
var middle = a[1..4];   // { 20, 30, 40 }
var head   = a[..3];    // { 10, 20, 30 }
var tail   = a[3..];    // { 40, 50, 60 }
```

In [ ]:
a = [10, 20, 30, 40, 50, 60]

print(a[1:4])    # [20, 30, 40] — index 1 up to (not including) 4
print(a[:3])     # first three — start defaults to 0
print(a[3:])     # from index 3 to the end — stop defaults to len(a)
print(a[:])      # everything (a shallow copy)

A helpful identity: for any `i`, `a[:i] + a[i:] == a`. The number of items in `a[i:j]` is `j - i` (when both are in range).

In [ ]:
i = 2
print(a[:i] + a[i:] == a)   # True
print(len(a[1:4]))          # 3

## Negative indexes

Negative positions count from the end, like C#'s `^n` "index from end" operator: `-1` is the last element, `-2` the second-to-last, and so on.

In [ ]:
a = [10, 20, 30, 40, 50, 60]

print(a[-2:])      # last two        — C#: a[^2..]
print(a[:-2])      # all but last 2  — C#: a[..^2]
print(a[1:-1])     # drop first and last — C#: a[1..^1]

⚠️ **Gotcha:** `-0` is just `0`, so `a[-0:]` is the **whole** list, not an empty one. In C#, `^0` means "the end", so the two don't line up. If you compute `n` dynamically, `a[-n:]` with `n == 0` returns everything.

In [ ]:
n = 0
print(a[-n:])          # the whole list, not []
print(a[len(a) - n:])  # [] — the safe way to say "last n items"

## The step

The optional third part is the **step** — take every *n*th element. C# ranges have no step; you'd use LINQ `Where((x, i) => i % 2 == 0)` or a loop.

In [ ]:
nums = list(range(10))     # [0, 1, ..., 9]

print(nums[::2])           # even positions: [0, 2, 4, 6, 8]
print(nums[1::2])          # odd positions:  [1, 3, 5, 7, 9]
print(nums[1:8:3])         # start 1, stop 8, every 3rd: [1, 4, 7]

## Negative steps and reversing

A negative step walks **backwards**. When the step is negative, the defaults flip: `start` defaults to the last element and `stop` to "before the first". The most common use is `[::-1]` to reverse.

```csharp
// C#
var reversed = a.Reverse().ToArray();     // LINQ
var text = new string("hello".Reverse().ToArray());
```

In [ ]:
nums = list(range(10))

print(nums[::-1])          # reversed copy
print(nums[::-2])          # every other, backwards: [9, 7, 5, 3, 1]
print(nums[7:2:-1])        # from 7 down to (not including) 2: [7, 6, 5, 4, 3]
print("hello"[::-1])       # works on strings too

⚠️ **Gotcha:** with a negative step, `start` must be *greater* than `stop`, otherwise you get an empty result — no error:

In [ ]:
print(nums[2:7:-1])   # [] — can't walk backwards from 2 to 7
print(nums[2:7])      # [2, 3, 4, 5, 6] — what was probably meant

A step of zero is the only slice that raises:

In [ ]:
try:
    nums[::0]
except ValueError as e:
    print(f"{type(e).__name__}: {e}")

## Out-of-range slicing doesn't raise

Indexing out of range raises `IndexError`, but **slicing** silently clamps the bounds to the sequence. In C#, `a[2..10]` on a 6-element array throws `ArgumentOutOfRangeException`.

In [ ]:
a = [10, 20, 30]

print(a[1:100])     # [20, 30] — stop clamped to len(a)
print(a[50:])       # [] — start past the end gives an empty list
print(a[-100:2])    # [10, 20] — start clamped to 0

try:
    a[100]
except IndexError as e:
    print(f"{type(e).__name__}: {e}")

This makes "first *n* items" safe without a length check: `items[:5]` works even when there are fewer than five items — like LINQ's `Take(5)`.

In [ ]:
short = ["x", "y"]
print(short[:5])    # ['x', 'y'] — C#: short.Take(5)

## Slicing strings and tuples

Slicing returns an object of the **same type**: a slice of a string is a string, a slice of a tuple is a tuple, a slice of a `range` is a `range`.

In [ ]:
filename = "report_2026.csv"
print(filename[:-4])        # 'report_2026'  — C#: filename[..^4]
print(filename[-3:])        # 'csv'
print(filename[7:11])       # '2026'

point = (1, 2, 3, 4)
print(point[1:3])           # (2, 3) — a tuple

r = range(0, 100, 5)
print(r[2:5])               # range(10, 25, 5) — still lazy

## Slices are (shallow) copies

Slicing a list creates a **new list** containing references to the same element objects — a *shallow* copy. Changing the slice doesn't affect the original list, but mutable elements are shared.

In C#, ranges on **arrays** and **strings** also copy (an array range calls `RuntimeHelpers.GetSubArray`). A `Span<T>` slice, however, is a **view** — writes through it change the original:

```csharp
// C#
int[] a = [1, 2, 3, 4];
int[] copy = a[1..3];              // new array
copy[0] = 99;                      // a unchanged

Span<int> view = a.AsSpan()[1..3]; // no copy
view[0] = 99;                      // a is now { 1, 99, 3, 4 }
```

In [ ]:
a = [1, 2, 3, 4]
part = a[1:3]
part[0] = 99
print(a, part)              # original unchanged

nested = [[1], [2], [3]]
part = nested[:2]
part[0].append("shared!")   # the inner lists are shared
print(nested)

Python's closest thing to `Span<T>` is `memoryview`, which works on binary buffers such as `bytearray` (not on lists). Slicing a `memoryview` does **not** copy:

In [ ]:
buffer = bytearray(b"hello world")
view = memoryview(buffer)[0:5]   # zero-copy view, like Span<byte>
view[0] = ord("J")
print(buffer)                    # bytearray(b'Jello world')

## Slice assignment (lists only)

Because lists are mutable, you can **assign to a slice**. The replacement can be a different length — the list grows or shrinks. There is no equivalent in C# ranges; you'd use `RemoveRange` + `InsertRange`.

```csharp
// C#
list.RemoveRange(1, 2);
list.InsertRange(1, ["b", "c", "d"]);
```

In [ ]:
letters = ["a", "x", "y", "e"]

letters[1:3] = ["b", "c", "d"]     # replace 2 items with 3
print(letters)

letters[1:1] = ["INSERTED"]        # empty slice = insert at position 1
print(letters)

letters[:] = ["fresh"]             # replace contents, keep the same list object
print(letters)

With a step other than 1 (an *extended* slice), the replacement must have exactly the same length:

In [ ]:
nums = [0, 1, 2, 3, 4, 5]
nums[::2] = ["a", "b", "c"]        # replace items at 0, 2, 4
print(nums)

try:
    nums[::2] = ["too", "few"]
except ValueError as e:
    print(f"{type(e).__name__}: {e}")

⚠️ **Gotcha:** `a[:] = ...` and `a = ...` are different. Slice assignment mutates the existing list, so every other reference to it sees the change; plain assignment just rebinds the name.

In [ ]:
data = [1, 2, 3]
alias = data

data = [9, 9]           # rebinds `data`; alias still has the old list
print(alias)

data = alias
data[:] = [9, 9]        # mutates the shared list
print(alias)

## Deleting slices

`del` works with slices too — `RemoveRange` in C#, but with step support.

In [ ]:
nums = list(range(10))
del nums[2:5]           # C#: nums.RemoveRange(2, 3)
print(nums)

del nums[::2]           # delete every other item
print(nums)

Strings and tuples are immutable, so slice assignment and deletion are not allowed on them:

In [ ]:
try:
    s = "hello"
    s[0:1] = "J"
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

## `slice` objects

`a[1:5:2]` is syntactic sugar for `a[slice(1, 5, 2)]`. You can store a `slice` in a variable and reuse it — handy for fixed-width records. It's the analogue of C#'s `System.Range` struct.

```csharp
// C#
Range year = 0..4;
var y = "2026-10-12"[year];
```

In [ ]:
YEAR = slice(0, 4)
MONTH = slice(5, 7)
DAY = slice(8, 10)

for date in ["2026-10-12", "2025-01-31"]:
    print(date[YEAR], date[MONTH], date[DAY])

`slice.indices(length)` resolves `None` and negative values into concrete `(start, stop, step)` numbers — useful when you implement slicing in your own class *(see Classes & OOP › 007 · Dunder Methods)*.

In [ ]:
s = slice(-3, None)
print(s)                 # slice(-3, None, None)
print(s.indices(10))     # (7, 10, 1)

## Summary

| Task | C# | Python |
|---|---|---|
| Sub-range | `a[1..4]` | `a[1:4]` |
| First n | `a[..n]` / `Take(n)` | `a[:n]` |
| Last n | `a[^n..]` / `TakeLast(n)` | `a[-n:]` (careful when `n == 0`) |
| Drop first & last | `a[1..^1]` | `a[1:-1]` |
| Every 2nd item | LINQ `Where((x, i) => i % 2 == 0)` | `a[::2]` |
| Reverse copy | `a.Reverse().ToArray()` | `a[::-1]` |
| Shallow copy | `a[..]` / `ToList()` | `a[:]` |
| Out of range | throws `ArgumentOutOfRangeException` | clamped silently |
| Zero-copy view | `Span<T>` / `Memory<T>` | `memoryview` (binary buffers only) |
| Replace range | `RemoveRange` + `InsertRange` | `a[i:j] = items` |
| Remove range | `RemoveRange(i, n)` | `del a[i:j]` |
| Reusable range | `Range r = 1..4;` | `r = slice(1, 4)` |

## Exercises

1. Given `s = "Python for C# developers"`, use slicing to get: the first word, the last word (`"developers"`), and the whole string reversed.
2. Check whether a word is a palindrome using a slice (ignore case): `"Racecar"`.
3. Given `nums = list(range(1, 11))`, replace the four items at positions 3 to 6 inclusive (the values `4, 5, 6, 7`) with a single `0` using slice assignment.
4. Rotate a list left by `k` positions with slicing: `[1, 2, 3, 4, 5]` with `k = 2` → `[3, 4, 5, 1, 2]`.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
# 1
s = "Python for C# developers"
first = s[:6]            # 'Python'  (or s.split()[0])
last = s[-10:]           # 'developers'
backwards = s[::-1]

# 2
word = "Racecar"
is_palindrome = word.lower() == word.lower()[::-1]   # True

# 3 — positions 3..6 inclusive is the slice 3:7
nums = list(range(1, 11))
nums[3:7] = [0]          # [1, 2, 3, 0, 8, 9, 10]

# 4
def rotate_left(items, k):
    if not items:
        return items[:]
    k %= len(items)
    return items[k:] + items[:k]

rotate_left([1, 2, 3, 4, 5], 2)   # [3, 4, 5, 1, 2]
```

</details>